In [4]:
if (!require("tidyverse")) install.packages("tidyverse")
library(tidyverse)

# --- Configuration ---
inputFolder <- "dataForRScripts"
output_dir <- "OutputImages"

# Input filename MUST match the Python output
inputFilename <- "Statistical_3km_Tavg_Precip_Incremental_Resumable.csv"
input_filepath <- file.path(inputFolder, inputFilename)

# Text for plot subtitles
baselineText <- "relative to 1995-2014 baseline"
downscalingText <- "Statistical (3km)"

# Create output directory if it doesn't exist
if (!dir.exists(output_dir)) {
  dir.create(output_dir)
}

# Define CMIP6 standard colors (Excluding SSP 1-2.6)
ssp_colors <- c("SSP 5-8.5" = "#840312", # dark red
                "SSP 3-7.0" = "#D55E00", # orangey red
                "SSP 2-4.5" = "#377eb8") # Blue

# --- Data Loading and Prep ---

# Check if input file exists and load the data
if (!file.exists(input_filepath)) {
  stop(paste("input file not found:", input_filepath, ". Run Python script first."))
}
print(paste("data read @:", input_filepath))

# read_csv
# Handle potential empty file if Python script failed completely
tryCatch({
  data <- read_csv(input_filepath, show_col_types = FALSE)
}, error = function(e) {
  stop(paste("Error reading CSV file:", e))
})


# Check if data is empty
if (nrow(data) == 0) {
  stop("CSV file is empty. Check Python script execution.")
}

# Convert categorical columns to factors
data <- data %>%
  mutate(Scenario = as.factor(Scenario),
         DataScenario = as.factor(DataScenario), # Important for grouping historical segments
         Simulation = as.factor(Simulation),
         Variable = as.factor(Variable),
         Region = as.factor(Region),
         Year = as.numeric(Year))

# Calculate ensemble means (multi-model means)
# Note: Ensemble mean calculation uses the 'Scenario' label (time-dependent)
ensembleMeans <- data %>%
  group_by(Year, Region, Scenario, Variable) %>%
  summarise(MeanAnomaly = mean(Anomaly, na.rm = TRUE), .groups = 'drop')

# --- Plot Functions ---

# Function to create time series plots
createTimeSeriesPLots <- function(dfIndividual, dfMeans, regionName, varName, yLab, plotTitle) {
  
  # Filter data
  plotdataIndividual <- dfIndividual %>% filter(Region == regionName, Variable == varName)
  plotDataMeans <- dfMeans %>% filter(Region == regionName, Variable == varName)
  
  # Separate historical data (based on 'Scenario' label)
  historicalIndividual <- plotdataIndividual %>% filter(Scenario == "Historical Climate")
  historicalMeans <- plotDataMeans %>% filter(Scenario == "Historical Climate")
  
  # Separate Future data (SSPs)
  future_individual <- plotdataIndividual %>% filter(Scenario != "Historical Climate")
  future_means <- plotDataMeans %>% filter(Scenario != "Historical Climate")

  # Start the plot
  p <- ggplot() +
    # 1. Historical individual sims (thin grey)
    # CRITICAL: Group by interaction(Simulation, DataScenario). This ensures that the historical 
    # segment of an SSP 2-4.5 run is plotted separately from the historical segment of an SSP 3-7.0 run.
    geom_line(data = historicalIndividual, aes(x = Year, y = Anomaly, group = interaction(Simulation, DataScenario)), 
              color = "grey", linewidth = 0.2, alpha = 0.6) +
    
    # 2. Future individual simulations (thin and colored)
    geom_line(data = future_individual, aes(x = Year, y = Anomaly, colour = Scenario, group = interaction(Scenario, Simulation)), 
              linewidth = 0.2, alpha = 0.5) +
    
    # 3. Historical ensemble mean (thick black line)
    geom_line(data = historicalMeans, 
              aes(x = Year, y = MeanAnomaly), 
              colour = "black", linewidth = 1.2) +
    
    # 4. Future ensemble means (thick and colored)
    geom_line(data = future_means, 
              aes(x = Year, y = MeanAnomaly, colour = Scenario), 
              linewidth = 1.5) +
    
    # Styling and labels
    geom_hline(yintercept = 0, linetype = "solid", color = "black", linewidth = 0.5)
    
    # Apply colour scale only to available scenarios
    available_colors <- ssp_colors[names(ssp_colors) %in% unique(future_individual$Scenario)]
    if (length(available_colors) > 0) {
        p <- p + scale_color_manual(values = available_colors, breaks = names(available_colors))
    }
    
    p <- p + labs(title = plotTitle,
         subtitle = paste(downscalingText, ",", baselineText),
         x = "Time (yr)",
         y = yLab) +   
    
    theme_bw(base_size = 14) + 
    theme(legend.position = "bottom",
          plot.title = element_text(hjust = 0.5, face = "bold"),
          plot.subtitle = element_text(hjust = 0.5),
          panel.grid.minor = element_blank()) +
    
    # X axis limits (1950-2100)
    scale_x_continuous(breaks = seq(1950, 2100, by = 20), limits = c(1950, 2100))

  return(p)
}

# --- Generate and Save Plots ---

regions <- levels(data$Region)
variables <- levels(data$Variable)

if (length(regions) == 0) {
    print("No regions found in data. Cannot generate plots.")
}

for (region in regions) {
  print(paste("Generating plots for region:", region))
  
  # Precipitation plot
  if ("Precip" %in% variables) {
    var_key <- "Precip"
    # Determine Y-axis label (Python calculates % change or Δmm/year if baseline is low)
    precip_data <- data %>% filter(Region == region, Variable == var_key)
    
    if (nrow(precip_data) > 0) {
        max_abs_anomaly <- max(abs(precip_data$Anomaly), na.rm = TRUE)
        
        # Heuristic: if max anomaly is very large (>500), it's likely absolute mm/year.
        if (is.finite(max_abs_anomaly) && max_abs_anomaly > 500) {
            y_label_precip <- "Precipitation Change (Δmm/year)"
        } else {
            y_label_precip <- "Precipitation Change (%)"
        }
        
        title <- paste(region, "- Decadal Mean Annual Precipitation Anomalies")
        pPrecip <- createTimeSeriesPLots(data, ensembleMeans, region, var_key, y_label_precip, title)
        
        # Save plot
        outputPrecipFileName <- file.path(output_dir, paste0(region, "_Statistical_3km_Precip_TimeSeries.png"))
        ggsave(outputPrecipFileName, plot = pPrecip, width = 10, height = 6, dpi = 300)
        print(paste("saved plot at:", outputPrecipFileName))
    } else {
        print(paste("  No precipitation data found for", region))
    }
  }
  
  # Temperature plot (T_Avg)
  if ("T_Avg" %in% variables) {
    var_key <- "T_Avg"
    temp_data <- data %>% filter(Region == region, Variable == var_key)
    if (nrow(temp_data) > 0) {
        # Python calculated absolute change (Δ°C)
        title <- paste(region, "- Decadal Mean Annual Avg Temperature (T_Avg) Anomalies")
        pTemp <- createTimeSeriesPLots(data, ensembleMeans, region, var_key, "Avg Temperature Change (Δ°C)", title)
        
        # Save plot
        output_filename_temp <- file.path(output_dir, paste0(region, "_Statistical_3km_Tavg_TimeSeries.png"))
        ggsave(output_filename_temp, plot = pTemp, width = 10, height = 6, dpi = 300)
        print(paste("Saved plot to:", output_filename_temp))
    } else {
        print(paste("  No T_Avg data found for", region))
    }
  }
}

print("R script finished.")

[1] "data read @: dataForRScripts/Statistical_3km_Tavg_Precip_Incremental_Resumable.csv"
[1] "Generating plots for region: JoshuaTree"
[1] "saved plot at: OutputImages/JoshuaTree_Statistical_3km_Precip_TimeSeries.png"
[1] "Saved plot to: OutputImages/JoshuaTree_Statistical_3km_Tavg_TimeSeries.png"
[1] "Generating plots for region: Mojave"
[1] "saved plot at: OutputImages/Mojave_Statistical_3km_Precip_TimeSeries.png"
[1] "Saved plot to: OutputImages/Mojave_Statistical_3km_Tavg_TimeSeries.png"
[1] "R script finished."


In [5]:
if (!require("tidyverse")) install.packages("tidyverse")
library(tidyverse)

# --- Configuration ---
inputFolder <- "dataForRScripts"
output_dir <- "OutputImages"

# Input filename MUST match the Python output
inputFilename <- "Statistical_3km_Tavg_Precip_Incremental_Resumable.csv"
input_filepath <- file.path(inputFolder, inputFilename)

# Text for plot subtitles
baselineText <- "relative to 1995-2014 baseline"
# Updated text to reflect the smoothing already applied by the Python pipeline
downscalingText <- "Statistical (3km), 10-yr rolling avg" 

# Create output directory if it doesn't exist
if (!dir.exists(output_dir)) {
  dir.create(output_dir)
}

# Define CMIP6 standard colors (Excluding SSP 1-2.6 as it was likely not processed)
ssp_colors <- c("SSP 5-8.5" = "#840312", # dark red
                "SSP 3-7.0" = "#D55E00", # orangey red
                "SSP 2-4.5" = "#377eb8") # Blue
historical_color <- "#636363" # Grey for historical

# Configuration for V3 (Bands)
# Use 1.64 standard deviations to approximate the 5%-95% confidence interval visualization
CI_MULTIPLIER <- 1.64

# --- Data Loading and Prep ---

# Check if input file exists and load the data
if (!file.exists(input_filepath)) {
  stop(paste("input file not found:", input_filepath, ". Run Python script first."))
}
print(paste("data read @:", input_filepath))

# read_csv
tryCatch({
  # The data loaded here is already smoothed (10-yr rolling avg) by the Python script.
  data <- read_csv(input_filepath, show_col_types = FALSE)
}, error = function(e) {
  stop(paste("Error reading CSV file:", e))
})

# Check if data is empty
if (nrow(data) == 0) {
  stop("CSV file is empty. Check Python script execution.")
}

# Convert categorical columns to factors
data <- data %>%
  mutate(Scenario = as.factor(Scenario),
         DataScenario = as.factor(DataScenario), # Important for grouping historical segments in V2
         Simulation = as.factor(Simulation),
         Variable = as.factor(Variable),
         Region = as.factor(Region),
         Year = as.numeric(Year))

# Calculate Ensemble Statistics (Used for V2 Mean line and V3 Ribbons)
ensembleStats <- data %>%
  group_by(Year, Region, Scenario, Variable) %>%
  summarise(
    MeanAnomaly = mean(Anomaly, na.rm = TRUE),
    StdDev = sd(Anomaly, na.rm = TRUE), # Standard deviation across models
    .groups = 'drop'
  ) %>%
  mutate(
    # Calculate the bounds using 1.64 * StdDev for V3
    LowerBound = MeanAnomaly - (CI_MULTIPLIER * StdDev),
    UpperBound = MeanAnomaly + (CI_MULTIPLIER * StdDev)
  )


# ==============================================================================
# VERSION 2 Function Definition: Standard Smoothed Lines (Individual Simulations)
# ==============================================================================

createTimeSeriesPlotsV2 <- function(dfIndividual, dfStats, regionName, varName, yLab, plotTitle) {
  
  # Filter data
  plotdataIndividual <- dfIndividual %>% filter(Region == regionName, Variable == varName)
  # Using MeanAnomaly from the stats dataframe
  plotDataMeans <- dfStats %>% filter(Region == regionName, Variable == varName)
  
  # Separate historical data
  historicalIndividual <- plotdataIndividual %>% filter(Scenario == "Historical Climate")
  historicalMeans <- plotDataMeans %>% filter(Scenario == "Historical Climate")
  
  # Separate Future data (SSPs)
  future_individual <- plotdataIndividual %>% filter(Scenario != "Historical Climate")
  future_means <- plotDataMeans %>% filter(Scenario != "Historical Climate")

  # Start the plot
  p <- ggplot() +
    # 1. Historical individual sims (thin grey)
    # Group by interaction(Simulation, DataScenario) to correctly plot historical segments of different SSP runs.
    geom_line(data = historicalIndividual, aes(x = Year, y = Anomaly, group = interaction(Simulation, DataScenario)), 
              color = "grey", linewidth = 0.2, alpha = 0.6) +
    
    # 2. Future individual simulations (thin and colored)
    geom_line(data = future_individual, aes(x = Year, y = Anomaly, colour = Scenario, group = interaction(Scenario, Simulation)), 
              linewidth = 0.2, alpha = 0.5) +
    
    # 3. Historical ensemble mean (thick black line)
    geom_line(data = historicalMeans, 
              aes(x = Year, y = MeanAnomaly), 
              colour = "black", linewidth = 1.2) +
    
    # 4. Future ensemble means (thick and colored)
    geom_line(data = future_means, 
              aes(x = Year, y = MeanAnomaly, colour = Scenario), 
              linewidth = 1.5) +
    
    # Styling and labels
    geom_hline(yintercept = 0, linetype = "solid", color = "black", linewidth = 0.5)
    
    # Apply colour scale only to available scenarios
    available_colors <- ssp_colors[names(ssp_colors) %in% unique(future_individual$Scenario)]
    if (length(available_colors) > 0) {
        p <- p + scale_color_manual(values = available_colors, breaks = names(available_colors))
    }
    
    p <- p + labs(title = plotTitle,
         subtitle = paste(downscalingText, ",", baselineText),
         x = "Time (yr)",
         y = yLab) +   
    
    theme_bw(base_size = 14) + 
    theme(legend.position = "bottom",
          plot.title = element_text(hjust = 0.5, face = "bold"),
          plot.subtitle = element_text(hjust = 0.5),
          panel.grid.minor = element_blank()) +
    
    # X axis limits (1950-2100)
    scale_x_continuous(breaks = seq(1950, 2100, by = 20), limits = c(1950, 2100))

  return(p)
}

# ==============================================================================
# VERSION 3 Function Definition: IPCC Style (Shaded Bands)
# ==============================================================================

createTimeSeriesPlotsV3 <- function(dfStats, regionName, varName, yLab, plotTitle) {
  
  # Filter data
  plotData <- dfStats %>% filter(Region == regionName, Variable == varName)
  
  # Separate historical and future
  historicalData <- plotData %>% filter(Scenario == "Historical Climate")
  futureData <- plotData %>% filter(Scenario != "Historical Climate")

  # Start the plot
  p <- ggplot() +
    
    # 1. Historical Uncertainty Ribbon (Grey)
    # Using LowerBound and UpperBound (5-95% CI)
    geom_ribbon(data = historicalData, 
                aes(x = Year, ymin = LowerBound, ymax = UpperBound),
                fill = historical_color, alpha = 0.3) +
    
    # 2. Future Uncertainty Ribbons (Colored by Scenario)
    geom_ribbon(data = futureData, 
                aes(x = Year, ymin = LowerBound, ymax = UpperBound, fill = Scenario),
                alpha = 0.3) + 

    # 3. Historical Ensemble Mean Line
    geom_line(data = historicalData, 
              aes(x = Year, y = MeanAnomaly), 
              color = historical_color, linewidth = 1.0) +
    
    # 4. Future Ensemble Mean Lines
    geom_line(data = futureData, 
              aes(x = Year, y = MeanAnomaly, color = Scenario), 
              linewidth = 1.0) +
    
    # Styling and labels
    geom_hline(yintercept = 0, linetype = "solid", color = "black", linewidth = 0.5)
    
    # Apply color scales (both fill and color need to be set)
    available_colors <- ssp_colors[names(ssp_colors) %in% unique(futureData$Scenario)]
    if (length(available_colors) > 0) {
        p <- p + scale_color_manual(values = available_colors, breaks = names(available_colors))
        p <- p + scale_fill_manual(values = available_colors, breaks = names(available_colors))
    }
    
    p <- p + labs(title = plotTitle,
         subtitle = paste(downscalingText, ",", baselineText),
         x = "Time (yr)",
         y = yLab) +   
    
    # Using a clean theme (theme_classic)
    theme_classic(base_size = 14) + 
    theme(legend.position = "bottom",
          legend.title = element_blank(),
          plot.title = element_text(hjust = 0.5, face = "bold"),
          plot.subtitle = element_text(hjust = 0.5),
          axis.line = element_line(color = "black"),
          panel.background = element_blank()) +
    
    scale_x_continuous(breaks = seq(1950, 2100, by = 25), limits = c(1950, 2100))

  return(p)
}


# ==============================================================================
# Generate and Save Plots (Both Versions)
# ==============================================================================

regions <- levels(data$Region)
variables <- levels(data$Variable)

if (length(regions) == 0) {
    print("No regions found in data. Cannot generate plots.")
}

for (region in regions) {
  print(paste("Generating plots for region:", region))
  
  # --- Precipitation Plots ---
  if ("Precip" %in% variables) {
    var_key <- "Precip"
    precip_data <- data %>% filter(Region == region, Variable == var_key)
    
    if (nrow(precip_data) > 0) {
        # Determine Y-axis label based on data characteristics (handles absolute vs relative change)
        max_abs_anomaly <- max(abs(precip_data$Anomaly), na.rm = TRUE)
        if (is.finite(max_abs_anomaly) && max_abs_anomaly > 500) {
            y_label_precip <- "Precipitation Change (Δmm/year)"
        } else {
            y_label_precip <- "Precipitation Change (%)"
        }
        
        title <- paste(region, "- Mean Annual Precipitation Anomalies")
        
        # Generate V2 (Lines)
        # We pass 'data' for individual lines and 'ensembleStats' for the mean line
        pPrecipV2 <- createTimeSeriesPlotsV2(data, ensembleStats, region, var_key, y_label_precip, title)
        # Naming convention as requested (V2 is the default name)
        outputFileNameV2 <- file.path(output_dir, paste0(region, "_Statistical_3km_Precip_TimeSeries.png"))
        ggsave(outputFileNameV2, plot = pPrecipV2, width = 10, height = 6, dpi = 300)
        print(paste("Saved V2 plot at:", outputFileNameV2))

        # Generate V3 (Bands)
        # We pass 'ensembleStats' for both the mean line and the ribbons
        pPrecipV3 <- createTimeSeriesPlotsV3(ensembleStats, region, var_key, y_label_precip, title)
        # Naming convention as requested (appending _v3)
        outputFileNameV3 <- file.path(output_dir, paste0(region, "_Statistical_3km_Precip_TimeSeries_v3.png"))
        ggsave(outputFileNameV3, plot = pPrecipV3, width = 10, height = 6, dpi = 300)
        print(paste("Saved V3 plot at:", outputFileNameV3))

    } else {
        print(paste("  No precipitation data found for", region))
    }
  }
  
  # --- Temperature Plots (T_Avg) ---
  if ("T_Avg" %in% variables) {
    var_key <- "T_Avg"
    temp_data <- data %>% filter(Region == region, Variable == var_key)
    if (nrow(temp_data) > 0) {
        y_label_temp <- "Avg Temperature Change (Δ°C)"
        title <- paste(region, "- Mean Annual Avg Temperature (T_Avg) Anomalies")

        # Generate V2 (Lines)
        pTempV2 <- createTimeSeriesPlotsV2(data, ensembleStats, region, var_key, y_label_temp, title)
        # Save as the standard name requested
        outputFileNameV2 <- file.path(output_dir, paste0(region, "_Statistical_3km_Tavg_TimeSeries.png"))
        ggsave(outputFileNameV2, plot = pTempV2, width = 10, height = 6, dpi = 300)
        print(paste("Saved V2 plot at:", outputFileNameV2))

        # Generate V3 (Bands)
        pTempV3 <- createTimeSeriesPlotsV3(ensembleStats, region, var_key, y_label_temp, title)
        # Save with _v3 appended
        outputFileNameV3 <- file.path(output_dir, paste0(region, "_Statistical_3km_Tavg_TimeSeries_v3.png"))
        ggsave(outputFileNameV3, plot = pTempV3, width = 10, height = 6, dpi = 300)
        print(paste("Saved V3 plot at:", outputFileNameV3))
        
    } else {
        print(paste("  No T_Avg data found for", region))
    }
  }
}

print("R script finished.")

[1] "data read @: dataForRScripts/Statistical_3km_Tavg_Precip_Incremental_Resumable.csv"
[1] "Generating plots for region: JoshuaTree"
[1] "Saved V2 plot at: OutputImages/JoshuaTree_Statistical_3km_Precip_TimeSeries.png"
[1] "Saved V3 plot at: OutputImages/JoshuaTree_Statistical_3km_Precip_TimeSeries_v3.png"
[1] "Saved V2 plot at: OutputImages/JoshuaTree_Statistical_3km_Tavg_TimeSeries.png"
[1] "Saved V3 plot at: OutputImages/JoshuaTree_Statistical_3km_Tavg_TimeSeries_v3.png"
[1] "Generating plots for region: Mojave"
[1] "Saved V2 plot at: OutputImages/Mojave_Statistical_3km_Precip_TimeSeries.png"
[1] "Saved V3 plot at: OutputImages/Mojave_Statistical_3km_Precip_TimeSeries_v3.png"
[1] "Saved V2 plot at: OutputImages/Mojave_Statistical_3km_Tavg_TimeSeries.png"
[1] "Saved V3 plot at: OutputImages/Mojave_Statistical_3km_Tavg_TimeSeries_v3.png"
[1] "R script finished."
